# Laboratorio 00.2 — Modelos de lenguaje multi-proveedor

Usaremos una interfaz común para preparar adaptadores de Ollama, OpenAI y Anthropic. Por defecto solo inspeccionaremos la configuración: ninguna celda hará una solicitud a un modelo hasta que se habilite de forma explícita.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Describir qué parte de una aplicación debe permanecer independiente del proveedor.
- Resolver el modelo desde un argumento, una variable de entorno o YAML.
- Preparar una comparación controlada entre proveedores con la misma pregunta.
- Registrar latencia, uso de tokens y fallos sin exponer credenciales.

## 2. Conceptos

Una abstracción de modelo separa la lógica de la aplicación del SDK específico del proveedor. `get_llm` construye adaptadores LangChain compatibles y deja configurables el proveedor, el modelo y la temperatura. Las credenciales pertenecen al entorno del proceso, no al código fuente. Cada proveedor puede devolver metadatos de uso distintos; una métrica no disponible debe registrarse como desconocida, no inventarse.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Aplicacion] --> B[get_llm]
    C[Argumentos] --> B
    D[Variables de entorno] --> B
    E[models.yaml] --> B
    B --> F[Adaptador Ollama]
    B --> G[Adaptador OpenAI]
    B --> H[Adaptador Anthropic]
    F --> I[Respuesta y metadatos]
    G --> I
    H --> I
```

## 4. Preparación del entorno

El siguiente bloque encuentra el paquete local y carga `.env` sin imprimir sus contenidos. La inspección posterior solo lee archivos de configuración; no necesita claves ni conectividad.

In [ ]:
import os

import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.models import get_llm, load_model_registry, load_runtime_settings

raiz_curso = preparar_entorno()
registro_modelos = load_model_registry(raiz_curso / 'config' / 'models.yaml')
ajustes = load_runtime_settings(raiz_curso / 'config' / 'settings.yaml')
print('Temperatura predeterminada:', ajustes.temperature)

## 5. Definición del problema

Queremos comparar respuestas a una misma pregunta sin reescribir la lógica de la aplicación para cada SDK. La prueba remota es opcional; el conjunto de datos inicial es la tabla de modelos declarados en YAML. No se enviará información privada ni se supondrá que todos los proveedores tienen las mismas capacidades.

## 6. Línea base

La línea base no usa un LLM: devuelve una respuesta determinista que confirma la recepción de la pregunta. Sirve para separar el cableado de una interfaz del comportamiento generativo.

In [ ]:
def respuesta_linea_base(pregunta: str) -> str:
    """Devuelve una respuesta local sin consumir una llamada a un modelo."""
    return f'Pregunta recibida: {pregunta}'

pregunta = '¿Qué diferencia hay entre validar y evaluar un sistema LLM?'
print(respuesta_linea_base(pregunta))

## 7. Implementación

Primero observamos los valores efectivos de configuración. Un modelo puede sobrescribirse por proveedor mediante `*_MODEL`; no mostramos claves secretas. La fábrica importa el SDK específico solo cuando se solicita ese proveedor.

In [ ]:
filas_modelos = []
for proveedor, configuracion in registro_modelos.providers.items():
    modelo = os.getenv(f'{proveedor.upper()}_MODEL', configuracion.model)
    filas_modelos.append({
        'proveedor': proveedor,
        'modelo': modelo,
        'requiere_clave': proveedor in {'openai', 'anthropic'},
        'requiere_servicio_local': proveedor == 'ollama',
    })

configuracion_proveedores = pd.DataFrame(filas_modelos).sort_values('proveedor')
display(configuracion_proveedores)

Para una comparación remota, cada fila de configuración representa como máximo una invocación. La siguiente función mide el tiempo y extrae tokens solo si el adaptador los informa; no calcula costos con tarifas implícitas. Los errores se registran por tipo, sin volcar detalles que pudieran incluir datos sensibles.

In [ ]:
from time import perf_counter

def ejecutar_comparacion(configuraciones: list[dict[str, object]], pregunta: str) -> pd.DataFrame:
    """Invoca una vez cada configuracion y registra metricas disponibles."""
    filas = []
    for configuracion in configuraciones:
        proveedor = str(configuracion['proveedor'])
        modelo = str(configuracion['modelo'])
        temperatura = float(configuracion.get('temperatura', ajustes.temperature))
        inicio = perf_counter()
        try:
            llm = get_llm(proveedor, model=modelo, temperature=temperatura)
            respuesta = llm.invoke(pregunta)
            duracion = perf_counter() - inicio
            uso = getattr(respuesta, 'usage_metadata', None) or {}
            uso_proveedor = getattr(respuesta, 'response_metadata', {}).get('token_usage', {})
            texto = respuesta.content
            filas.append({
                'proveedor': proveedor,
                'modelo': modelo,
                'temperatura': temperatura,
                'latencia_segundos': duracion,
                'tokens_entrada': uso.get('input_tokens', uso_proveedor.get('prompt_tokens')),
                'tokens_salida': uso.get('output_tokens', uso_proveedor.get('completion_tokens')),
                'costo_estimado': None,
                'respuesta': texto if isinstance(texto, str) else str(texto),
                'estado': 'exito',
                'tipo_error': None,
            })
        except Exception as error:
            filas.append({
                'proveedor': proveedor,
                'modelo': modelo,
                'temperatura': temperatura,
                'latencia_segundos': perf_counter() - inicio,
                'tokens_entrada': None,
                'tokens_salida': None,
                'costo_estimado': None,
                'respuesta': None,
                'estado': 'error',
                'tipo_error': type(error).__name__,
            })
    return pd.DataFrame(filas)

## 8. Experimentos

Inspecciona primero el plan de ejecución. Para comparar proveedores mantén fijos pregunta y temperatura; para comparar temperaturas conserva proveedor, modelo y pregunta. Una configuración vacía y el interruptor desactivado garantizan cero llamadas por defecto.

In [ ]:
CONFIGURACIONES_PRUEBA: list[dict[str, object]] = []
EJECUTAR_LLAMADAS = False
llamadas_estimadas = len(CONFIGURACIONES_PRUEBA)
print(f'Llamadas máximas planeadas: {llamadas_estimadas}')
if not EJECUTAR_LLAMADAS:
    print('Sin solicitudes: habilita el interruptor solo después de revisar el costo y las credenciales.')
elif not CONFIGURACIONES_PRUEBA:
    raise ValueError('Agrega configuraciones de prueba antes de habilitar las llamadas.')
else:
    resultados = ejecutar_comparacion(CONFIGURACIONES_PRUEBA, pregunta)
    display(resultados)

## 9. Evaluación

La evaluación informa latencia, cobertura de uso de tokens y tasa de fallos. Si todavía no ejecutaste solicitudes, el tamaño cero se muestra explícitamente en lugar de fabricar métricas. Calidad de respuesta y costo quedan sin puntuar hasta contar con un evaluador y tarifas verificables.

In [ ]:
if 'resultados' not in globals():
    resultados = pd.DataFrame(columns=[
        'proveedor', 'modelo', 'temperatura', 'latencia_segundos',
        'tokens_entrada', 'tokens_salida', 'costo_estimado', 'respuesta', 'estado', 'tipo_error',
    ])

cantidad = len(resultados)
metricas_comparacion = pd.DataFrame([{
    'solicitudes': cantidad,
    'tasa_fallos': (resultados['estado'] == 'error').mean() if cantidad else None,
    'latencia_media_segundos': resultados['latencia_segundos'].mean() if cantidad else None,
    'cobertura_tokens_entrada': resultados['tokens_entrada'].notna().mean() if cantidad else None,
    'cobertura_tokens_salida': resultados['tokens_salida'].notna().mean() if cantidad else None,
}])
display(metricas_comparacion)
configuracion_proveedores.to_csv(raiz_curso / 'outputs' / '00_02_provider_configurations.csv', index=False)
if cantidad:
    resultados.to_csv(raiz_curso / 'outputs' / '00_02_model_trials.csv', index=False)

## 10. Discusión

La abstracción reduce cambios en el código de aplicación, pero no elimina diferencias de disponibilidad, capacidades, límites, metadatos o precio entre servicios. La ausencia de tokens o costo es información faltante, no un cero. Una respuesta aislada tampoco permite concluir que un proveedor sea universalmente superior.

## 11. Ejercicios

1. Cambia un modelo por argumento y luego por variable de entorno; verifica la precedencia de cada valor.
2. Añade dos temperaturas para el mismo proveedor y compara únicamente las métricas que realmente estén disponibles.
3. Registra la disponibilidad del servidor Ollama sin mostrar credenciales ni enviar una pregunta.
4. Define una política de retención para las respuestas guardadas en los CSV.

## 12. Desafío

Diseña un comparador que evalúe varias preguntas con un límite duro de llamadas, un presupuesto estimado configurable y una parada inmediata cuando se alcance cualquiera de los límites. Añade pruebas usando adaptadores simulados, sin consumir tokens.

## 13. Ideas clave

- La lógica de aplicación debe depender de una interfaz estable, no de un SDK concreto.
- Configuración, secretos y ejecución son responsabilidades distintas.
- Cada llamada remota debe ser deliberada y contable.
- Compara modelos con el mismo experimento y comunica los datos que no están disponibles.